# Benchmark: pandas vs Polars, operation by operation

- 1_demo.ipynb times the whole query.
- 2_benchmark.ipynb times its operations one by one.

The data is read once at the start, so each line times the operation only.
Polars is timed in lazy mode, collect() included.

In [1]:
import tempfile
from pathlib import Path

import pandas as pd
import polars as pl

DATA = Path("data")  # the data/ folder next to this notebook
MEASUREMENTS_FILE = DATA / "measurements.parquet"
STATIONS_FILE = DATA / "stations.parquet"
STATION = "ber"  # Bern / Zollikofen
COLUMNS = ["station_abbr", "ts", "temperature"]
OUT = Path(tempfile.gettempdir()) / "benchmark_out.parquet"  # to time the write

print(f"pandas {pd.__version__}, polars {pl.__version__}")

pandas 3.0.6, polars 1.44.2


### Read the data only at start

In [2]:
pandas_df = pd.read_parquet(MEASUREMENTS_FILE)
pandas_stations = pd.read_parquet(STATIONS_FILE)
polars_df = pl.read_parquet(MEASUREMENTS_FILE)
polars_stations = pl.read_parquet(STATIONS_FILE)

print(f"{polars_df.height:,} rows")

3,831,715 rows


### Read Parquet

In [3]:
read_pandas = %timeit -o -n 1 -r 10 pd.read_parquet(MEASUREMENTS_FILE)
read_polars = %timeit -o -n 1 -r 10 pl.scan_parquet(MEASUREMENTS_FILE).collect()

130 ms ± 13.1 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)
40.3 ms ± 3.05 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


### Select columns

In [4]:
select_pandas = %timeit -o -n 1 -r 10 pandas_df[COLUMNS]
select_polars = %timeit -o -n 1 -r 10 polars_df.lazy().select(COLUMNS).collect()

1.35 ms ± 533 μs per loop (mean ± std. dev. of 10 runs, 1 loop each)
The slowest run took 19.30 times longer than the fastest. This could mean that an intermediate result is being cached.
96.2 μs ± 168 μs per loop (mean ± std. dev. of 10 runs, 1 loop each)


### Filter rows

In [5]:
filter_pandas = %timeit -o -n 1 -r 10 pandas_df[pandas_df["station_abbr"] == STATION]
filter_polars = %timeit -o -n 1 -r 10 polars_df.lazy().filter(pl.col("station_abbr") == STATION).collect()

39.7 ms ± 13.5 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)
3.46 ms ± 913 μs per loop (mean ± std. dev. of 10 runs, 1 loop each)


### Join stations

In [6]:
join_pandas = %timeit -o -n 1 -r 10 pandas_df.merge(pandas_stations, on="station_abbr")
join_polars = %timeit -o -n 1 -r 10 polars_df.lazy().join(polars_stations.lazy(), on="station_abbr").collect()

1.17 s ± 106 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)
47.1 ms ± 5.46 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


### Group + mean

In [7]:
group_pandas = %timeit -o -n 1 -r 10 pandas_df.groupby("station_abbr")["temperature"].mean()
group_polars = %timeit -o -n 1 -r 10 polars_df.lazy().group_by("station_abbr").agg(pl.col("temperature").mean()).collect()

115 ms ± 15.3 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)
The slowest run took 5.06 times longer than the fastest. This could mean that an intermediate result is being cached.
17.7 ms ± 13.5 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


### Sort

In [8]:
sort_pandas = %timeit -o -n 1 -r 10 pandas_df.sort_values("ts")
sort_polars = %timeit -o -n 1 -r 10 polars_df.lazy().sort("ts").collect()

374 ms ± 21.6 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)
53.3 ms ± 10.1 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


### Write Parquet

In [9]:
write_pandas = %timeit -o -n 1 -r 10 pandas_df.to_parquet(OUT)
write_polars = %timeit -o -n 1 -r 10 polars_df.write_parquet(OUT)

OUT.unlink()  # delete the file written by the test

756 ms ± 85.9 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)
96.5 ms ± 5.26 ms per loop (mean ± std. dev. of 10 runs, 1 loop each)


### The table

`.best` is the fastest of the 10 runs, in seconds. `* 1000` turns it into milliseconds.

In [10]:
table = pl.DataFrame({
    "operation": ["read Parquet", "select columns", "filter rows", "join stations",
                  "group + mean", "sort", "write Parquet"],
    "pandas (ms)": [read_pandas.best, select_pandas.best, filter_pandas.best, join_pandas.best,
                    group_pandas.best, sort_pandas.best, write_pandas.best],
    "Polars (ms)": [read_polars.best, select_polars.best, filter_polars.best, join_polars.best,
                    group_polars.best, sort_polars.best, write_polars.best],
})

table = table.with_columns(
    (pl.col("pandas (ms)") * 1000).round(2),
    (pl.col("Polars (ms)") * 1000).round(2),
)
table = table.with_columns((pl.col("pandas (ms)") / pl.col("Polars (ms)")).round(1).alias("ratio"))
table

operation,pandas (ms),Polars (ms),ratio
str,f64,f64,f64
"""read Parquet""",116.72,36.12,3.2
"""select columns""",0.92,0.03,30.7
"""filter rows""",29.02,2.56,11.3
"""join stations""",1069.26,41.75,25.6
"""group + mean""",92.59,11.49,8.1
"""sort""",348.77,45.81,7.6
"""write Parquet""",635.84,89.41,7.1
